# HW3: Regression and Classification

In this assignment you will preprocess the dataset and perform some basic regression and classification tasks. The learning outcome of this part is to know how one can pre-process a real-world dataset and perform a supervised learning task, and to understand some of the fundamental mechanisms behind these tasks.

##  Grading:

Pass/Fail.

To Pass this HW you need to provide a complete and correct solution, passing all the tests.

## OUTLINE:

Data pre-processing, regression task and classification task

1. Reading the files
2. Missing Values
3. Imputing categorical variables
4. Imputing numerical variables
5. Classification with Decision Tree, single split
6. Classification with Decision Tree, Cross validation
7. Interpretation of the results

## Important instructions:

Each function you make will be considered during the grading, so it is important to strictly follow input and output instructions stated in the skeleton code.

You must not change the names of the functions, since, if you do, the tests will fail.

Since this Homework is, in part, focused on having you implement creative solutions to impute missing data, if at any point of the homework you will use functions like fillna(), SimpleImputer(), IterativeImputer(), or packages like fancyimpute, missingpy, or similar, you will fail a test designed to spot these packages. Please, try to avoid circumventing this rule, since een if you manage to pass the homework, a similar task might be in the exam, and there you would be spotted for sure.

## Homework Scenario: Cleaning and Preparing Heart Disease Data

You have recently joined the **Data Science and Analytics Unit** at the *Global Health Institute (GHI)*, a non-profit organization focused on improving cardiovascular disease diagnosis through data-driven research.  

A junior data analyst from your team, **Franco**, sends you a message:

> “Hey, welcome to the team! We’re preparing a predictive model to help doctors identify patients at risk of heart disease using clinical data from several hospitals.  
>   
> We have two related datasets:
> - **Cleveland dataset** → this will be used for **training and validation**
> - **Hungary dataset** → this will serve as our **independent test set**
>
> Unfortunately, it looks like something went wrong during the data collection process: some values appear to have been **corrupted or lost**. Before we can train any classification model, we need to **inspect and clean the data**, handle **missing or inconsistent values**, and make sure it’s ready for modeling. I'm completely lost and I have a lot of other work, can you please help me with the cleaning and with creating some baselines classification models?”

Your task is to **analyze and clean the datasets** before **building a classifier** to predict whether a patient has heart disease.

In [ ]:
# these are the libraries that you will need throughout the assignment
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from matplotlib.colors import ListedColormap

from HW import *

RSEED = 8

## *1.* Reading the files

### `Task: Read the datasets from the 'datasets' folder. Use the files called cleveland.csv and hungary.csv that you have downloaded in this archive.`

## Heart Disease Dataset — Column Descriptions

Someone has changed the names of some columns in the dataset, so make sure to use this description and refer to it for the "allowed" values.

Common sense is useful when evaluating some of the features: for example, in this dataset there is no column called weight, but, if there was one, since we are talking about humans and not ethereal beings, if a patient had a value of 0 in the weight column, this value could be due to a typo, or corrupted, and would need to be cleaned in some way.

| **Column** | **Description** |
|-------------|-----------------|
| **Age** | Age of the patient (in years). This dataset only includes adult patients. |
| **Sex** | Biological sex of the patient: `1 = male`, `0 = female`. |
| **ChestPainType** | Type of chest pain experienced: <br>• `1` = typical angina <br>• `2` = atypical angina <br>• `3` = non-anginal pain <br>• `4` = asymptomatic. |
| **RestBP** | Resting blood pressure (in mm Hg) measured on admission to the hospital. |
| **Chol** | Serum cholesterol level (in mg/dl). |
| **FBS** | Fasting blood sugar: `1` if fasting blood sugar > 120 mg/dl, otherwise `0`. |
| **RestECG** | Resting electrocardiographic results: <br>• `0` = normal <br>• `1` = ST-T wave abnormality <br>• `2` = showing probable or definite left ventricular hypertrophy. |
| **MaxHR** | Maximum heart rate achieved during the exercise test. |
| **ExAng** | Exercise-induced angina: `1` = yes, `0` = no. |
| **Oldpeak** | ST depression induced by exercise relative to rest (a measure of exercise-induced ischemia). |
| **Slope** | Slope of the peak exercise ST segment: <br>• `1` = upsloping <br>• `2` = flat <br>• `3` = downsloping. |
| **Ca** | Number of major vessels (0–3) colored by fluoroscopy (a measure of blood flow). |
| **Thal** | Thalassemia test result: <br>• `3` = normal <br>• `6` = fixed defect <br>• `7` = reversible defect. |
| **Num** | Diagnosis of heart disease (target variable): <br>`0` = no heart disease, `1–4` = presence of heart disease with increasing severity. |


In [ ]:
from sklearn.impute import KNNImputer
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np

In [ ]:
# From the folder 'datasets', read the files cleveland.csv and hungary.csv into the dataframes cleveland and test, respectively.

cleveland = pd.read_csv('/content/cleveland.csv')  # change this
test = pd.read_csv('/content/hungary.csv')       # change this

In [ ]:
cleveland.columns

In [ ]:
test.columns

In [ ]:
# You can uncomment this to inspact the datasets
cleveland.head(5)

In [ ]:
(cleveland['Ca']=='?').sum()

In [ ]:
(cleveland['Thal']=='?').sum()

In [ ]:
test.head(5)

In [194]:
# if you want to see information about the dataset, uncomment:
cleveland.describe()

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Num
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000
mean,55.052805,0.693069,3.442244,131.623762,3543.082508,0.148515,1.277228,246.953795,0.343234,1.039604,1.597360,0.937294
std,16.236574,0.599270,5.080393,17.549467,57434.537877,0.356198,5.210380,1715.122158,0.540737,1.161075,0.622104,1.228536
min,-1.000000,-1.000000,1.000000,94.000000,-234.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000
25%,47.000000,0.000000,3.000000,120.000000,211.000000,0.000000,0.000000,132.500000,0.000000,0.000000,1.000000,0.000000
50%,56.000000,1.000000,3.000000,130.000000,240.000000,0.000000,1.000000,152.000000,0.000000,0.800000,2.000000,0.000000
75%,61.000000,1.000000,4.000000,140.000000,274.500000,0.000000,2.000000,166.000000,1.000000,1.600000,2.000000,2.000000
max,222.000000,7.000000,90.000000,200.000000,1000000.000000,1.000000,90.000000,30000.000000,5.000000,6.200000,3.000000,4.000000


In [ ]:
# if you want to see information about the dataset, uncomment:
test.describe()

In [ ]:
cleveland.isna().sum()

In [ ]:
test.isna().sum()

In [ ]:
test.shape

In [ ]:
cleveland.dtypes

In [ ]:
test.dtypes

In [ ]:
(test['Slope'] == '?').sum()

In [ ]:
(test['Ca'] == '?').sum()

## *2.* Missing values

### `Task: use the function clean_data from the HW.py file to get a clean version of the cleveland and test dataframes.`

In [211]:
def clean_data(df):
    """
    Cleans the Cleveland heart disease dataset strictly based on the provided description:
    - Replaces '?' with NaN
    - Converts all columns to numeric dtype
    - Replaces invalid/out-of-spec values with NaN:
        * categorical columns outside allowed values
        * continuous columns with negative values
    - Returns cleaned DataFrame and missing value counts
    """

    df = df.copy()

    categorical_columns = [
        'Sex', 'ChestPainType', 'FBS', 'RestECG',
        'ExAng', 'Slope', 'Ca', 'Thal'
    ]
    numerical_columns = [
        'Age', 'RestBP', 'Chol', 'MaxHR', 'Oldpeak'
    ]

    # All ? with NAN first
    df = df.replace('?', np.nan)

    # All are numeric column with values later on category column by its nature
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    valid_values = {
        'Sex': [0, 1],
        'ChestPainType': [1, 2, 3, 4],
        'FBS': [0, 1],
        'RestECG': [0, 1, 2],
        'ExAng': [0, 1],
        'Slope': [1, 2, 3],
        'Ca': [0, 1, 2, 3],
        'Thal': [3, 6, 7],
        'Num': [0, 1, 2, 3, 4]
    }
    column_including_target_as_categorical = categorical_columns + ['Num']
    for col in column_including_target_as_categorical:
        if col in df.columns:
            df.loc[~df[col].isin(valid_values[col]), col] = np.nan

    for col in numerical_columns:
        if col in df.columns:
            if col == 'Oldpeak':
            # Oldpeak can be 0, but not negative
                df.loc[df[col] < 0, col] = np.nan
            elif col == 'Age':
                # Age can't be <= 0 or > 100
                df.loc[(df[col] < 18) | (df[col] > 100), col] = np.nan
            elif col == 'Chol':
                df.loc[(df[col] <=0) | (df[col] >= 600), col] = np.nan
            elif col == 'MaxHR':
                df.loc[(df[col] <=0) | (df[col] >= 220), col] = np.nan
            else:
                # For other continuous features, 0 or negative are invalid
                df.loc[df[col] <= 0, col] = np.nan

        missing_values_count = df.isna().sum().to_dict()

    return df, missing_values_count


In [196]:
(cleveland['Chol']<=0).sum()

np.int64(2)

In [212]:
# Write your code here
# cleveland_cleaned, missing_values_cleveland = pd.DataFrame(), {} # change this
# test_cleaned, missing_values_test = pd.DataFrame(), {} # change this

cleveland_cleaned, missing_values_cleveland = clean_data(cleveland) # change this
test_cleaned, missing_values_test = clean_data(test) # change this
print(missing_values_test)
print(missing_values_cleveland)

{'Age': 4, 'Sex': 1, 'ChestPainType': 0, 'RestBP': 0, 'Chol': 25, 'FBS': 8, 'RestECG': 2, 'MaxHR': 4, 'ExAng': 1, 'Oldpeak': 0, 'Slope': 188, 'Ca': 289, 'Thal': 264, 'Num': 0}
{'Age': 5, 'Sex': 2, 'ChestPainType': 1, 'RestBP': 0, 'Chol': 3, 'FBS': 0, 'RestECG': 2, 'MaxHR': 3, 'ExAng': 1, 'Oldpeak': 0, 'Slope': 1, 'Ca': 5, 'Thal': 3, 'Num': 0}


In [ ]:
test_cleaned.isna().sum()

In [ ]:
cleveland_cleaned.columns

## *3.* Imputing categorical variables

At the beginning of this file you can find the names of the columns and a description of their contents.

Determine which columns are categorical, and set their type to object.

Determine which columns are numerical, and set their type accordingly.

Do not include the target column in any of these lists!

In [ ]:
categorical_columns = [
        'Sex', 'ChestPainType', 'FBS', 'RestECG',
        'ExAng', 'Slope', 'Ca', 'Thal'
    ]
numerical_columns = [
        'Age', 'RestBP', 'Chol', 'MaxHR', 'Oldpeak'
    ]

target = 'Num'

In [ ]:
cleveland_cleaned['Age'].min()

### ` Task: Split the cleveland_cleaned dataframe in a train and a validation set, using train_test_split from sklearn. `

The train set must be called train, the validation set must be called val. The size of the validation set must be 30% of the total size of the cleveland_cleaned dataframe. Use shuffle=True and stratify the split based on y_cleveland. Make sure that both train and val are dataframes, and that the columns have the correct names. Reset the indexes of all four the dataframes, using drop=True.

In [ ]:
# Split the data into X and y, where X contains the features and y contains the target variable.
X_cleveland = cleveland_cleaned.drop(columns=['Num'])  # change this
y_cleveland = cleveland_cleaned['Num']  # change this

X_test = test_cleaned.drop(columns=['Num'])     # change this
y_test = test_cleaned['Num']      # change this

# For test data, just reset index
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_cleveland,
    y_cleveland,
    test_size=0.3,
    shuffle=True,
    stratify=y_cleveland,
    random_state=RSEED
)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)


In [ ]:
print(type(X_train))

In [ ]:
# # if you want to see information about the split dataset, uncomment:
X_train.head(5)

In [ ]:
# # if you want to see information about the split dataset, uncomment:
X_val.head(5)

In [ ]:
X_train.min()

In [ ]:
# Changing y to 0 and 1 only
y_train = pd.DataFrame([0 if i == 0 else 1 for i in y_train], columns=['Num'])
y_val   = pd.DataFrame([0 if i == 0 else 1 for i in y_val], columns=['Num'])
y_test  = pd.DataFrame([0 if i == 0 else 1 for i in y_test], columns=['Num'])

In [ ]:
X_train.describe()

### ` Task: use the impute_missing_categorical function from the HW.py file to impute the missing data from the categorical features in your dataframes. `

In [ ]:
# Task 3: Categorical Features Imputation

def impute_missing_categorical(df_train, df_val, df_test, categorical_columns):
    """
    Task: Categorical Features Imputation
    --------------------------------------
    This function should handle missing values in categorical columns using appropriate techniques.
    We will skip scaling and encoding to keep things simple.

    Instructions:
    - Create subsets of the input DataFrames (train, validation, test) with only the categorical columns.
    - Use KNNImputer with k=5 and weights set to 'distance' to fill missing values in the categorical columns.
    - Ensure that the imputed values are approximated to the nearest value in the original dataset for each column, to avoid artifacts like decimal values.
    - If any "new value" is equidistant from two original values, choose the smaller one.
    - Add the column names to the resulting DataFrames after imputation.
    - The imputed dataframes should only contain the categorical columns.

    Parameters:
    df_train (pd.DataFrame): The training DataFrame.
    df_val (pd.DataFrame): The validation DataFrame.
    df_test (pd.DataFrame): The test DataFrame.
    categorical_columns (list): A list of column names corresponding to categorical features.

    Returns:
    pd.DataFrame: The training DataFrame with imputed categorical features.
    pd.DataFrame: The validation DataFrame with imputed categorical features.
    pd.DataFrame: The test DataFrame with imputed categorical features.
    """

    X_train_cat = df_train[categorical_columns].copy()
    X_val_cat = df_val[categorical_columns].copy()
    X_test_cat = df_test[categorical_columns].copy()

    imputer = KNNImputer(n_neighbors=5, weights='distance')
    imputer.fit(X_train_cat)

    X_train_imputed = imputer.transform(X_train_cat)
    X_val_imputed = imputer.transform(X_val_cat)
    X_test_imputed = imputer.transform(X_test_cat)

    for i, col in enumerate(categorical_columns):
        valid_values = np.sort(df_train[col].dropna().unique())  # original values
        # Function to snap each imputed value to nearest valid value
        def snap_to_nearest(value):
            diffs = np.abs(valid_values - value)
            min_diff = diffs.min()
            nearest_values = valid_values[diffs == min_diff]
            return nearest_values.min()  # pick smaller if tie
        X_train_imputed[:, i] = np.vectorize(snap_to_nearest)(X_train_imputed[:, i])
        X_val_imputed[:, i] = np.vectorize(snap_to_nearest)(X_val_imputed[:, i])
        X_test_imputed[:, i] = np.vectorize(snap_to_nearest)(X_test_imputed[:, i])

    # # --- Step 5: Convert back to DataFrames with correct column names ---
    X_train_imputed = pd.DataFrame(X_train_imputed, columns=categorical_columns)
    X_val_imputed = pd.DataFrame(X_val_imputed, columns=categorical_columns)
    X_test_imputed = pd.DataFrame(X_test_imputed, columns=categorical_columns)

    return X_train_imputed, X_val_imputed, X_test_imputed



In [ ]:
# Write your code here
X_train_imputed_cat, X_val_imputed_cat, X_test_imputed_cat = impute_missing_categorical(X_train, X_val, X_test, categorical_columns)

In [ ]:
X_train_imputed_cat['Sex'].unique()

In [ ]:
X_test_imputed_cat.isna().sum()

## *4.* Imputing numerical variables

` Task: use the impute_missing_numeric function from the HW.py file to impute the missing data from the numeric features in your dataframes. `

In [ ]:
def impute_numerical_features(df_train, df_val, df_test, numerical_columns):
    """
    Iterative Lasso-based imputer for numerical features.
    Handles NaNs internally by temporarily imputing other columns used in prediction.
    Fills missing values in train, val, and test using only models trained on non-missing train data for each numeric column.
    Always imputes the column with the fewest missing values next.
    Uses a fallback mean imputation on any remaining missing values.
    """

    train_num = df_train[numerical_columns].copy()
    val_num = df_val[numerical_columns].copy()
    test_num = df_test[numerical_columns].copy()

    train_idx, val_idx, test_idx = train_num.index, val_num.index, test_num.index

    train_imputed = train_num.copy()
    val_imputed = val_num.copy()
    test_imputed = test_num.copy()

    while (train_imputed.isnull().any().any() or val_imputed.isnull().any().any() or test_imputed.isnull().any().any()):
        missing_counts = train_imputed.isnull().sum()
        cols_with_missing = missing_counts[missing_counts > 0]
        if len(cols_with_missing) == 0:
            break

        col_to_impute = cols_with_missing.sort_values().index[0]
        not_missing = train_imputed[col_to_impute].notnull()

        X_train_fit = train_imputed.loc[not_missing].drop(columns=[col_to_impute])
        y_train_fit = train_imputed.loc[not_missing, col_to_impute]

        # Fill missing predictors temporarily
        X_train_fit = X_train_fit.fillna(X_train_fit.mean())

        if len(X_train_fit) > 0:
            model = Lasso(alpha=0.01, max_iter=2000)
            model.fit(X_train_fit.values, y_train_fit.values)

            for df in [train_imputed, val_imputed, test_imputed]:
                missing_mask = df[col_to_impute].isnull()
                if missing_mask.any():
                    X_missing = df.loc[missing_mask].drop(columns=[col_to_impute]).fillna(X_train_fit.mean())
                    if len(X_missing) > 0:
                        df.loc[missing_mask, col_to_impute] = model.predict(X_missing.values)

    # Fallback mean imputation
    for col in numerical_columns:
        col_mean = train_imputed[col].mean()
        train_imputed[col].fillna(col_mean, inplace=True)
        val_imputed[col].fillna(col_mean, inplace=True)
        test_imputed[col].fillna(col_mean, inplace=True)

    return train_imputed.loc[train_idx], val_imputed.loc[val_idx], test_imputed.loc[test_idx]


In [ ]:
# Impute numerical features using iterative Lasso
X_train_imputed_num, X_val_imputed_num, X_test_imputed_num = impute_numerical_features(
    df_train=X_train,
    df_val=X_val,
    df_test=X_test,
    numerical_columns=numerical_columns
)

In [ ]:
X_val_imputed_num.isna().sum()

### ` Task: use the merge_imputed function from the HW.py file to merge your imputed dataframes. `

In [ ]:
def merge_imputed(df_cat, df_num):
    """
    Task: Merge Imputed DataFrames
    -------------------------------
    This function should merge the imputed categorical and numerical DataFrames.

    Instructions:
    - Merge the imputed categorical and numerical DataFrames on their indexes.
    - Ensure that the resulting DataFrame contains all columns from both input DataFrames.

    Parameters:
    df_cat (pd.DataFrame): The DataFrame with imputed categorical features.
    df_num (pd.DataFrame): The DataFrame with imputed numerical features.

    Returns:
    pd.DataFrame: The merged DataFrame containing both categorical and numerical features.
    """
    merged = pd.concat([df_cat, df_num], axis=1)
    return merged

In [ ]:
# Merge the train_imputed_cat and train_imputed_num datasets. Call the resulting dataset X_train_imputed.
# Merge the val_imputed_cat and val_imputed_num datasets. Call the resulting dataset X_val_imputed.
# Merge the test_imputed_cat and test_imputed_num datasets. Call the resulting dataset X_test_imputed.

# Write your code here
# Merge imputed numerical and categorical datasets by columns
X_train_imputed = merge_imputed(X_train_imputed_cat, X_train_imputed_num)
X_val_imputed = merge_imputed(X_val_imputed_cat, X_val_imputed_num)
X_test_imputed = merge_imputed(X_test_imputed_cat, X_test_imputed_num)


In [ ]:
X_test_imputed.isna().sum()

## *5.* Classification, using a single split

### ` Use the function train_and_evaluate_single_split to produce classification results for your test set.`

In [ ]:
# Task 5: Classification Using a Single Split
def train_and_evaluate_single_split(X_train, X_val, y_train, y_val, cat_cols,num_cols,model, hp):
    """
    Task: Classification Using a Single Split
    ------------------------------------------
    This function should train a classification pipeline on the training set and evaluate it on the validation set, using the provided parameters.

    Instructions:
    - Create a classification pipeline. It should include:
        - A OneHotEncoder for categorical features (handle_unknown='ignore').
        - A StandardScaler for numerical features.
        - The provided classification model.
    - Use ColumnTransformer to apply the appropriate transformations to categorical and numerical features.
    - Set the model parameters using the provided parameters dictionary.
    - Train the model using the training data (X_train, y_train).
    - Evaluate the model on the validation data (X_val, y_val) using F1 score.
    - Return the evaluation results (F1 score) for the given parameters combination.

    Parameters:
    X_train (pd.DataFrame): The training feature set.
    X_val (pd.DataFrame): The validation feature set.
    y_train (pd.Series): The training labels.
    y_val (pd.Series): The validation labels.
    model: The classification model to train.
    hp (dict): A dictionary of hyperparameters to set for the model.

    Returns:
    dict: A dictionary containing two keys: 'params' (training parameters) and 'F1 scores' (F1 score). Each key should have the correct value.
    """
    model.set_params(**hp)

    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ])

    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('classifier', model)
    ])


    pipeline.fit(X_train, y_train.values.ravel())


    y_pred = pipeline.predict(X_val)

    f1 = f1_score(y_val, y_pred)

    return {'params': hp, 'F1 scores': f1}

In [ ]:
# The hyperparameters for the tree should be:
# criterion: ['gini', 'entropy']
# max_depth: [3, 5, 10]
# The hyperparameters for the logistic regression should be:
# penalty: ['l1', 'l2']
# C: [0.1, 10]
# solver: ['liblinear']

# For each combination of hyperparameters, train a classification pipeline using your function.


from sklearn.model_selection import ParameterGrid
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import time

hyperparameters_tree = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 10]
}
hyperparameters_logreg = {
    'penalty': ['l1', 'l2'],
    'C': [0.1, 10],
    'solver': ['liblinear']
}
performance_df = pd.DataFrame(columns=['params', 'F1 scores'])

# create a list from the grid of hyperparameters for each model, and create the models.

start = time.time() # DO NOT CHANGE/DELETE THIS LINE

for number in range(1, 11): # change this
    # call your function here, then concat the results to performance_df
    # --- Decision Tree ---
    for criterion in hyperparameters_tree['criterion']:
        for max_depth in hyperparameters_tree['max_depth']:
            hp = {'criterion': criterion, 'max_depth': max_depth}
            dt_model = DecisionTreeClassifier()
            result = train_and_evaluate_single_split(
                X_train_imputed, X_val_imputed, y_train, y_val, categorical_columns, numerical_columns,
                model=dt_model,
                hp=hp
            )
            performance_df = pd.concat([performance_df, pd.DataFrame([result])], ignore_index=True)

    # --- Logistic Regression ---
    for penalty in hyperparameters_logreg['penalty']:
        for C in hyperparameters_logreg['C']:
            for solver in hyperparameters_logreg['solver']:
                hp = {'penalty': penalty, 'C': C, 'solver': solver}
                lr_model = LogisticRegression()
                result = train_and_evaluate_single_split(
                    X_train_imputed, X_val_imputed, y_train, y_val, categorical_columns, numerical_columns,
                    model=lr_model,
                    hp=hp
                )
                performance_df = pd.concat([performance_df, pd.DataFrame([result])], ignore_index=True)

end = time.time() # DO NOT CHANGE/DELETE THIS LINE

print('Time elapsed to run the hyperparameter tuning with a single split: ', end - start) # DO NOT CHANGE/DELETE THIS LINE

In [ ]:
performance_df.sort_values(by='F1 scores', ascending=False)

In [ ]:
performance_df.iloc[16].params

In [ ]:
# 	{'criterion': 'entropy', 'max_depth': 3}	0.657534

In [ ]:
# {'penalty': 'l2', 'C': 0.1, 'solver': 'liblinear'}	0.810127

In [ ]:
# Concatenate the train and validation datasets. Call the resulting datasets X and y.
# X = pd.DataFrame()  # change this
# y = pd.DataFrame()  # change this

X = pd.concat([X_train_imputed, X_val_imputed], ignore_index=True)
# Concatenate targets
y = pd.concat([y_train, y_val], ignore_index=True)
# retrain the model with the best hyperparameters on the whole training dataset.
# Remember to use the same preprocessing steps as before.

# Write your code here
best_hp = {'penalty': 'l1', 'C': 0.1, 'solver': 'liblinear'}  # replace with actual best
best_model = LogisticRegression(**best_hp)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
        ('num', StandardScaler(), numerical_columns)
    ]
)
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', best_model)
])
y = y.squeeze()
pipeline.fit(X, y)

## *6.* Classification with Decision Tree using Cross Validation

### ` Use the function train_and_evaluate_cross_validation to produce classification results for your test set.`

In [ ]:
X.head()

In [ ]:
# Task 6: Classification Using Cross-Validation
def train_and_evaluate_cross_validation(X, y, cat_cols,num_cols,model, hp, cv):
    """
    Task: Classification Using Cross-Validation
    --------------------------------------------
    This function should train and evaluate a classification model using cross-validation.

    Instructions:
    - Use cross-validation to train and evaluate the model, with shuffle set to True and using the specified number of folds (cv).
    - For each fold, create a classification pipeline similar to the one in Task 5.
    - Evaluate the model on each fold using F1 score.
    - Return the average F1 score across all folds for each parameter combination.
    - Ensure that the cross-validation process is reproducible (use random_state = 8).

    Parameters:
    X (pd.DataFrame): The feature set.
    y (pd.Series): The labels.
    model: The classification model to train.
    hp (dict): A dictionary of hyperparameters to set for the model.
    cv (int): The number of cross-validation folds.

    Returns:
    dict: A dictionary containing two keys: 'params' (training parameters) and 'Average F1 scores' (F1 score). Each key should have the correct value.
    """

    f1_scores = []

    # Set model hyperparameters
    model.set_params(**hp)

    # ColumnTransformer for preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
            ('num', StandardScaler(), num_cols)
        ]
    )

    # Pipeline
    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])

    # Stratified K-Fold
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=8)

    for train_idx, val_idx in skf.split(X, y):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]


        y_train_fold = y_train_fold.squeeze()
        y_val_fold = y_val_fold.squeeze()

        pipeline.fit(X_train_fold, y_train_fold)
        y_pred = pipeline.predict(X_val_fold)

        f1_scores.append(f1_score(y_val_fold, y_pred))

    return {
        'params': hp,
        'Average F1 scores': np.mean(f1_scores)
    }

In [ ]:
# 1. Use the same hyperparameters from the previous task.
# 2. Create a dataframe to store the performance of the model with cross-validation, containing the columns 'params' and 'Average F1 scores'
# 3. You can reuse the parameter grids from the previous step.
# 4. Run your function for each combination of hyperparameters, using 5-fold cross-validation.
# 5. Concatenate the results to the dataframe created in step 2.


# DO NOT FORGET TO DELETE THE PREVIOUS LINES. They are only to make the empty assignment run without errors,
# but they will destroy the data you need.

performance_df_cv = pd.DataFrame(columns=['params', 'Average F1 scores'])

start_CV = time.time() # DO NOT CHANGE/DELETE THIS LINE

# call your function here, then concat the results to performance_df_cv
# --- Decision Tree ---
for criterion in hyperparameters_tree['criterion']:
    for max_depth in hyperparameters_tree['max_depth']:
        hp = {'criterion': criterion, 'max_depth': max_depth}
        dt_model = DecisionTreeClassifier()
        result = train_and_evaluate_cross_validation(X, y, categorical_columns, numerical_columns,dt_model, hp, cv=5)
        performance_df_cv = pd.concat([performance_df_cv, pd.DataFrame([result])], ignore_index=True)

# --- Logistic Regression ---
for penalty in hyperparameters_logreg['penalty']:
    for C in hyperparameters_logreg['C']:
        for solver in hyperparameters_logreg['solver']:
            hp = {'penalty': penalty, 'C': C, 'solver': solver}
            lr_model = LogisticRegression()
            result = train_and_evaluate_cross_validation(X, y, categorical_columns, numerical_columns,lr_model, hp, cv=5)
            performance_df_cv = pd.concat([performance_df_cv, pd.DataFrame([result])], ignore_index=True)


end_CV = time.time() # DO NOT CHANGE/DELETE THIS LINE

print('Time elapsed to run the hyperparameter tuning with Cross Validation: ', end_CV - start_CV) # DO NOT CHANGE/DELETE THIS LINE


In [ ]:
performance_df_cv.sort_values(by='Average F1 scores',ascending=False)

In [ ]:
performance_df_cv.iloc[6].params

In [ ]:
# retrain the model with the best hyperparameters on the whole training dataset.
# Remember to use the same preprocessing steps as before.

# Concatenate the train and validation datasets. Call the resulting datasets X and y.
# X = pd.DataFrame()  # change this
# y = pd.DataFrame()  # change this

X = pd.concat([X_train_imputed, X_val_imputed], ignore_index=True)
# Concatenate targets
y = pd.concat([y_train, y_val], ignore_index=True)
# retrain the model with the best hyperparameters on the whole training dataset.
# Remember to use the same preprocessing steps as before.

# Write your code here
best_hp = {'penalty': 'l1', 'C': 0.1, 'solver': 'liblinear'}  # replace with actual best
best_model = LogisticRegression(**best_hp)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
        ('num', StandardScaler(), numerical_columns)
    ]
)
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', best_model)
])
y = y.squeeze()
pipeline.fit(X, y)

## *7.* Interpretation of the results

### ` Which model performs the best? `

Logistic Regression with parameters : {'penalty': 'l1', 'C': 0.1, 'solver': 'liblinear'} gived the best performance when used with Stratified K fold method on X and y

### ` Task: use the best model to produce predictions on the test set, then calculate the F1 score on the test set. What do you notice? `

In [ ]:
y_test_pred = pipeline.predict(X_test_imputed)
f1_score(y_test, y_test_pred)

### ` What is a possible explanation? `